# Bài 28: Hình học epipolar

So khớp ảnh nổi ở Bài 22 giả định một cặp ảnh *đã nắn thẳng*, nơi các điểm tương ứng luôn rơi trên cùng một hàng. Trong trường hợp tổng quát hơn, các camera ảnh nổi có thể không được nắn thẳng. Bài này trình bày mối quan hệ hai góc nhìn tổng quát giữa **bất kỳ** cặp ảnh nào của một cảnh tĩnh. Như ta sẽ thấy, điểm khớp của một điểm trong ảnh kia không chỉ nằm *ở đâu đó* &mdash; nó bị ràng buộc phải nằm trên một đường thẳng cụ thể. Ta trình bày cả trường hợp đã hiệu chỉnh lẫn chưa hiệu chỉnh.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Ràng buộc epipolar

Với một điểm $x_1$ trong ảnh 1 và điểm khớp thực $x_2$ của nó trong ảnh 2 (cả hai đều ở tọa độ điểm ảnh thuần nhất), tồn tại một ma trận $3\times3$ thiếu hạng $F$ &mdash; gọi là **ma trận cơ bản (fundamental matrix)** &mdash; sao cho

$$x_2^\top F x_1 = 0$$

với *mọi* cặp tương ứng, bất kể hình học của cảnh. $F$ chỉ phụ thuộc vào tư thế tương đối của hai camera và các tham số nội (chưa hiệu chỉnh). Sắp xếp lại, $l_2 = F x_1$ là **đường epipolar (epipolar line)** trong ảnh 2: đường thẳng 1D mà điểm khớp của $x_1$ chắc chắn nằm trên đó. Đây chính là cơ chế khiến việc tìm kiếm giới hạn theo hàng ở Bài 22 hợp lệ &mdash; nắn thẳng chỉ là cách bố trí camera đặc biệt mà ở đó mọi đường epipolar tình cờ nằm ngang, thu gọn việc tìm kiếm 1D về một hàng duy nhất.

### Tại sao lại là một đường thẳng? Một hình ảnh hình học

Camera 1 chỉ biết *hướng* tới một điểm, không biết độ sâu của nó: điểm 3D thực $X$ có thể ở bất kỳ đâu dọc theo tia đi từ $C_1$ qua $x_1$. Nhưng mọi điểm trên tia đó, cùng với $C_1$ và $C_2$, đều nằm trong một mặt phẳng duy nhất &mdash; gọi là **mặt phẳng epipolar (epipolar plane)**. Camera 2 nhìn mặt phẳng này từ cạnh bên, thành một đường thẳng duy nhất: dù $X$ thực sự nằm ở đâu dọc theo tia, hình chiếu của nó vào ảnh 2 luôn rơi trên đường thẳng đó, chính là đường epipolar.

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

C1 = np.array([0.0, 0.0, 0.0])
C2 = np.array([3.0, 0.5, -0.5])
ray_dir = np.array([0.6, 0.8, 3.0])
ray_dir = ray_dir / np.linalg.norm(ray_dir)
depths = [2.0, 3.5, 5.0]                        # a few candidate depths along camera 1's ray
candidates = [C1 + d * ray_dir for d in depths]

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection='3d')

ax.plot(*zip(C1, C2), c='gray', linestyle='--', linewidth=1)
for X in candidates:
    ax.plot(*zip(C1, X), c='tab:blue', linewidth=1)
    ax.plot(*zip(C2, X), c='tab:green', linewidth=1)
cand_arr = np.array(candidates)
ax.scatter(cand_arr[:, 0], cand_arr[:, 1], cand_arr[:, 2], c='black', s=30)
ax.text(*candidates[-1], '  possible $X$\n  (unknown depth)', fontsize=8)

ax.scatter(*C1, c='tab:blue', s=60)
ax.text(*C1, '  $C_1$', fontsize=10)
ax.scatter(*C2, c='tab:green', s=60)
ax.text(*C2, '  $C_2$', fontsize=10)

plane = Poly3DCollection([[C1, C2, candidates[-1]]], alpha=0.15, facecolor='tab:orange')
ax.add_collection3d(plane)
centroid = (C1 + C2 + candidates[1]) / 3
ax.text(*centroid, 'epipolar\nplane', fontsize=8, color='tab:orange', ha='center')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Why the match is constrained to a line')
plt.tight_layout()
plt.show()

Ba đoạn màu lam đều là các tia từ $C_1$ qua cùng một điểm ảnh $x_1$, chỉ được kéo dài tới ba độ sâu khác nhau, đều hợp lý như nhau &mdash; riêng camera 1 không thể phân biệt chúng. Nhưng mỗi ứng viên, cùng với $C_1$ và $C_2$, vẫn nằm trong một mặt phẳng epipolar được hiển thị, nên hình chiếu của chúng vào camera 2 (các đầu mút của các đoạn màu xanh lục) đều rơi dọc theo giao tuyến của mặt phẳng đó với ảnh 2: chính là đường epipolar. $F$ chỉ là bộ công cụ đại số dự đoán trực tiếp đường thẳng này từ $x_1$, mà không bao giờ cần biết độ sâu.

Ảnh động này minh họa sự mơ hồ về độ sâu bằng cách vẽ trực tiếp mặt phẳng ảnh của cả hai camera trong cảnh 3D để mối quan hệ tia-với-đường-epipolar hiện ra ngay trong một cái nhìn: khi độ sâu thay đổi, hình chiếu của điểm lên camera 2 luôn nằm trên đường epipolar.

<img src="../animations/epipolar_depth_demo.gif" alt="Ảnh động một điểm 3D trượt dọc theo tia của camera 1, với mặt phẳng ảnh của cả hai camera được vẽ trong cảnh: x1 cố định ở tâm mặt phẳng ảnh 1, và hình chiếu của điểm trượt dọc theo đường epipolar cố định trên mặt phẳng ảnh 2" style="display: block; margin: 0 auto; max-width: 560px;">

## Một cảnh hai camera tổng hợp

Ta dựng hai camera với tham số nội $K$ đã biết và phép xoay/tịnh tiến tương đối đã biết, tạo một số điểm 3D ngẫu nhiên, chiếu các điểm vào cả hai camera, và dùng các cặp tương ứng thu được để khôi phục $F$.

In [ ]:
rng = np.random.default_rng(0)
K = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]], dtype=np.float64)

R1, t1 = np.eye(3), np.zeros(3)                       # camera 1: at the origin, looking down +z
angle = np.radians(15)
R_true = np.array([[np.cos(angle), 0, np.sin(angle)],
                    [0, 1, 0],
                    [-np.sin(angle), 0, np.cos(angle)]])
t_true = np.array([0.5, 0.0, 0.1])                    # camera 2: rotated 15 deg, shifted along x

P1 = K @ np.hstack([R1, t1.reshape(3, 1)])
P2 = K @ np.hstack([R_true, t_true.reshape(3, 1)])

def project(P, points_3d):
    homogeneous = np.hstack([points_3d, np.ones((len(points_3d), 1))])
    projected = (P @ homogeneous.T).T
    return projected[:, :2] / projected[:, 2:3]

points_3d = rng.uniform(-1, 1, (40, 3)) + np.array([0, 0, 5])  # in front of both cameras
x1 = project(P1, points_3d)
x2 = project(P2, points_3d)

print(f'{len(points_3d)} 3D points, projected into both cameras')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(x1[:, 0], x1[:, 1], c='tab:blue', s=15)
axes[0].set_xlim(0, 640); axes[0].set_ylim(480, 0)
axes[0].set_title('Image 1')
axes[1].scatter(x2[:, 0], x2[:, 1], c='tab:orange', s=15)
axes[1].set_xlim(0, 640); axes[1].set_ylim(480, 0)
axes[1].set_title('Image 2')
plt.tight_layout()
plt.show()

### SVD: Phân tích giá trị suy biến

Ở Bài 23, ta đã thấy rằng một ma trận thực đối xứng nửa xác định dương $A$ (như ma trận hiệp phương sai) có thể viết dưới dạng $A = P \Lambda P^\top$, với $P$ chứa các *vector riêng*, và $\Lambda$ là ma trận chéo không âm chứa các *giá trị riêng*. **Phân tích giá trị suy biến (singular value decomposition — SVD)** tổng quát hóa ý tưởng này: mọi ma trận $A$ &mdash; đối xứng hay không, vuông hay không &mdash; đều có thể viết dưới dạng $A = U \Sigma V^\top$, với $U$ và $V$ trực giao chứa các **vector suy biến (singular vectors)** (lần lượt là trái và phải) và $\Sigma$ là ma trận chéo không âm chứa các **giá trị suy biến (singular values)**. Theo định nghĩa, các giá trị suy biến trong $\Sigma$ được sắp xếp từ lớn đến nhỏ. Với mục đích của ta, ta tập trung vào hai sự thật cụ thể về SVD:

**Giải $Ax=0$ gần đúng nhất có thể, với ràng buộc $\|x\|=1$.** Đáp án là cột cuối cùng của $V$ (tương đương, hàng cuối cùng của $V^\top$) &mdash; vector suy biến phải ứng với giá trị suy biến nhỏ nhất. Đây là mẹo bình phương tối thiểu toàn phần ở Bài 23, được tổng quát hóa: các cột của $V$ là các vector riêng của $A^\top A$, sắp theo giá trị riêng. Nên hướng ứng với giá trị suy biến nhỏ nhất ở đây chính xác là cùng "hướng có sai số khớp nhỏ nhất" như vector riêng ứng với giá trị riêng nhỏ nhất mà Bài 23 đã dùng.

**Áp đặt ràng buộc hạng.** Đưa (các) giá trị suy biến nhỏ nhất về 0 rồi tái tạo sẽ cho ra ma trận hạng thấp hơn gần nhất (theo nghĩa bình phương tối thiểu) với ma trận gốc. Đó chính là điều cần thiết để ép $F$ ước lượng được &mdash; vốn phải có hạng chính xác bằng 2 &mdash; trở lại ràng buộc đó, sau khi phép giải 9 tham số không ràng buộc chắc chắn trôi khỏi nó một chút.

## Thuật toán 8 điểm

Mỗi cặp tương ứng cho một phương trình tuyến tính theo 9 phần tử chưa biết của $F$ (khai triển $x_2^\top F x_1 = 0$). $F$ chỉ được xác định sai khác một hệ số tỉ lệ (giống phép đồng nhất ở Bài 24), nên nó thực sự chỉ có 8 ẩn số độc lập, giải được như một bài toán bình phương tối thiểu thuần nhất qua SVD (ở trên) &mdash; cùng bộ công cụ DLT đã dùng để khớp phép đồng nhất ở Bài 24. Vì $F$ phải có hạng chính xác bằng 2 (ma trận cơ bản luôn suy biến), ta chiếu nghiệm 9 tham số trở lại ma trận hạng 2 gần nhất, dùng mẹo SVD ở trên.

(Lưu ý rằng $F$ thực ra chỉ có 7 bậc tự do thực sự, vì ràng buộc hạng 2 loại bỏ thêm một bậc; tuy nhiên, bất kỳ thuật toán 7 điểm nào cũng sẽ phức tạp hơn vì phải tính đến một ràng buộc phi tuyến; thuật toán 8 điểm hay ở chỗ nó là một thuật toán *tuyến tính*, dù cần một bước riêng để khôi phục tính chất hạng 2.)

Còn một trục trặc nữa trước khi ta có thể giao bài toán này cho SVD: tọa độ điểm ảnh thô (cỡ hàng trăm) làm cho hệ tuyến tính có điều kiện số rất kém. Vì vậy trước tiên ta co giãn lại các điểm của mỗi ảnh sao cho có tâm tại gốc tọa độ với khoảng cách trung bình $\sqrt2$ tới gốc, giải trong các tọa độ chuẩn hóa này, rồi hoàn tác phép co giãn trên $F$ thu được. Bước cải thiện điều kiện này, đã được chứng minh tạo ra khác biệt đáng kể trong thực tế, là thứ biến thuật toán 8 điểm đơn thuần thành **thuật toán 8 điểm chuẩn hóa (normalized 8-point algorithm)** (<a href="../references.html#hartley-1997">Hartley, 1997</a>) &mdash; phiên bản được dùng bên dưới, và là phiên bản nên dùng trong thực tế.

In [ ]:
def normalize_points(x):
    """Shift/scale so points are centered at the origin with average distance sqrt(2) -- standard
    numerical-conditioning trick (Hartley normalization) for the 8-point algorithm."""
    mean = x.mean(axis=0)
    std = x.std()
    T = np.array([[1 / std, 0, -mean[0] / std],
                  [0, 1 / std, -mean[1] / std],
                  [0, 0, 1]])
    x_h = np.hstack([x, np.ones((len(x), 1))])
    return (T @ x_h.T).T, T

def eight_point_algorithm(x1, x2):
    x1n, T1 = normalize_points(x1)
    x2n, T2 = normalize_points(x2)

    A = np.array([[xb * xa, xb * ya, xb, yb * xa, yb * ya, yb, xa, ya, 1]
                  for (xa, ya, _), (xb, yb, _) in zip(x1n, x2n)])
    _, _, Vt = np.linalg.svd(A)
    F = Vt[-1].reshape(3, 3)

    U, S, Vt2 = np.linalg.svd(F)   # enforce rank-2 by zeroing the smallest singular value
    S[-1] = 0
    F = U @ np.diag(S) @ Vt2

    F = T2.T @ F @ T1              # undo the normalization
    return F / F[2, 2]

F_mine = eight_point_algorithm(x1, x2)
F_cv, _ = cv2.findFundamentalMat(x1, x2, cv2.FM_8POINT)

print('our F:\n', np.round(F_mine, 5))
print('cv2.findFundamentalMat F:\n', np.round(F_cv, 5))

### Kiểm tra trực tiếp ràng buộc epipolar

Với các cặp tương ứng thực, $x_2^\top F x_1$ phải bằng 0 (về mặt số).

In [ ]:
def epipolar_residual(F, x1, x2):
    x1h = np.hstack([x1, np.ones((len(x1), 1))])
    x2h = np.hstack([x2, np.ones((len(x2), 1))])
    return np.abs(np.sum(x2h * (F @ x1h.T).T, axis=1))

print(f'mean |x2^T F x1|, our F:  {epipolar_residual(F_mine, x1, x2).mean():.2e}')
print(f'mean |x2^T F x1|, cv2 F:  {epipolar_residual(F_cv, x1, x2).mean():.2e}')

### Trực quan hóa các đường epipolar

Với một nhúm điểm trong ảnh 1, ta vẽ các đường epipolar $l_2 = Fx_1$ của chúng trong ảnh 2, và xác nhận mỗi điểm khớp thực nằm chính xác trên đường của nó.

In [ ]:
w, h = 640, 480
sample_idx = rng.choice(len(x1), 6, replace=False)
colors = plt.cm.tab10.colors[:len(sample_idx)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(x1[:, 0], x1[:, 1], c='gray', s=15)
axes[0].scatter(x1[sample_idx, 0], x1[sample_idx, 1], c=colors, s=40)
axes[0].set_xlim(0, w); axes[0].set_ylim(h, 0)
axes[0].set_title('Image 1: 6 selected points')

axes[1].scatter(x2[:, 0], x2[:, 1], c='gray', s=15)
for i, color in zip(sample_idx, colors):
    a, b, c = F_mine @ np.array([x1[i, 0], x1[i, 1], 1])   # line: a*x + b*y + c = 0
    xs = np.array([0, w])
    ys = -(a * xs + c) / b
    axes[1].plot(xs, ys, linewidth=1, color=color)
axes[1].scatter(x2[sample_idx, 0], x2[sample_idx, 1], c=colors, s=40, zorder=5,
                edgecolors='black', linewidths=0.6, label='true match')
axes[1].set_xlim(0, w); axes[1].set_ylim(h, 0)
axes[1].set_title('Image 2: epipolar lines + true matches')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

Mọi điểm khớp thực đều rơi chính xác trên đường dự đoán của nó &mdash; việc tìm kiếm một điểm trong ảnh thứ hai thực sự thu gọn từ 2D xuống 1D, ngay cả khi không nắn thẳng các ảnh trước.

## Từ ma trận cơ bản đến ma trận thiết yếu: bổ sung hiệu chỉnh

Ma trận cơ bản làm việc trên tọa độ điểm ảnh thô, giả định các tham số nội của camera chưa biết. Nếu ta *có* biết tham số nội $K$ (từ hiệu chỉnh camera &mdash; Bài 27), ta có thể loại bỏ chúng và làm việc trong tọa độ chuẩn hóa, cho ra **ma trận thiết yếu (essential matrix)**:

$$E = K_2^\top F K_1 \qquad \text{(if both are the same camera, then } K_1 = K_2 = K \text{)}$$

Khác với $F$ có 7 bậc tự do, $E$ chỉ có 5 bậc tự do &mdash; nó được xây dựng hoàn toàn từ một phép xoay tương đối $R$ và một hướng tịnh tiến $t$ giữa hai camera: $E = [t]_\times R$, trong đó $[t]_\times$ là "ma trận tích có hướng" phản đối xứng $3\times3$ của $t$, được định nghĩa sao cho $[t]_\times v = t \times v$ với mọi vector $v$. Điều này có nghĩa là $E$ có thể được *phân rã* ngược lại thành $R$ và $t$, đó là cách khôi phục chuyển động camera chỉ từ các cặp tương ứng trong ảnh.

Ta ước lượng $E$ một cách bền vững bằng `cv2.findEssentialMat(..., method=cv2.RANSAC, ...)`, chính là loại phép khớp loại bỏ ngoại lai đã xây dựng từ đầu ở Bài 23.

In [ ]:
E_from_F = K.T @ F_mine @ K
E_direct, _ = cv2.findEssentialMat(x1, x2, K, method=cv2.RANSAC, threshold=1.0)

# E is also only defined up to scale -- compare directions, not raw magnitudes
print('E derived from our F (normalized):\n', np.round(E_from_F / np.linalg.norm(E_from_F), 4))
print('E from cv2.findEssentialMat (normalized):\n', np.round(E_direct / np.linalg.norm(E_direct), 4))

### Khôi phục chuyển động camera

`cv2.recoverPose` phân rã $E$ thành một phép xoay và một *hướng* tịnh tiến &mdash; không phải một vector tịnh tiến đầy đủ, vì về bản chất độ lớn không thể khôi phục được chỉ từ hai góc nhìn. Một cảnh lớn gấp đôi, được nhìn bởi các camera có đường cơ sở giữa chúng gấp đôi, tạo ra các ảnh giống hệt nhau; đây cũng là sự mơ hồ về tỉ lệ quen thuộc trong thị giác đơn mắt (monocular).

In [ ]:
_, R_estimated, t_estimated, _ = cv2.recoverPose(E_direct, x1, x2, K)

print('true rotation:\n', np.round(R_true, 4))
print('recovered rotation:\n', np.round(R_estimated, 4))
print()
print('true translation direction:      ', np.round(t_true / np.linalg.norm(t_true), 4))
print('recovered translation direction: ', np.round(t_estimated.ravel(), 4))

Phép xoay được khôi phục, và hướng tịnh tiến khớp sai khác dấu/tỉ lệ như kỳ vọng. Chỉ từ các cặp tương ứng đặc trưng (Bài 20), ta đã khôi phục được phép xoay và tịnh tiến tương đối (sai khác tỉ lệ) giữa hai camera. Đây là điểm khởi đầu cho cấu trúc từ chuyển động (Bài 29).

### Bài tập

1. Thêm nhiễu điểm ảnh (ví dụ độ lệch chuẩn 0.5) vào `x1` và `x2` trước khi chạy thuật toán 8 điểm. Phần dư epipolar trung bình tăng bao nhiêu, và việc chuẩn hóa tọa độ (như `eight_point_algorithm` làm) có thực sự quan trọng ở đây không &mdash; hãy thử bỏ qua bước chuẩn hóa và so sánh.
2. Điểm epipole (epipole) trong ảnh 2 là hình chiếu của tâm camera 1, và thỏa mãn $F e_1 = 0$ với epipole $e_1$ trong ảnh 1 (và $F^\top e_2 = 0$ với epipole $e_2$ trong ảnh 2). Hãy tính cả hai epipole dưới dạng không gian rỗng (null space) của $F$ (qua SVD) và kiểm tra xem chúng nằm trong hay ngoài vùng ảnh nhìn thấy được với cấu hình camera này.
3. Tăng góc xoay giữa hai camera lên 60 độ và chạy lại phép khôi phục tư thế. `cv2.recoverPose` có còn tìm ra phép xoay đúng không? Đến mức nào thì bạn kỳ vọng chính việc so khớp tương ứng (Bài 20) mới trở thành nút thắt cổ chai thay vì hình học?